# Imports

In [10]:
import os
import urllib.request
import zipfile
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm


# Local Directory Setup

In [14]:
# Base Directory
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')

# --- NEW ARCHITECTURE ---
DATASETS_DIR = os.path.join(DATA_DIR, 'Datasets')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
REDUCED_DIR = os.path.join(DATA_DIR, 'Features_Reduced')
RESULTS_DIR = os.path.join(DATA_DIR, 'Results')

# Create all necessary directories
for d in [DATA_DIR, DATASETS_DIR, RAW_DIR, REDUCED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Local Data Directory set to: {DATA_DIR}")
print("Subdirectories created: Datasets/, Features_RAW/, Features_Reduced/, Results/")

# CRITICAL: GitHub Security (.gitignore)
gitignore_path = os.path.join(os.getcwd(), '.gitignore')
with open(gitignore_path, 'a+') as f:
    f.seek(0)
    content = f.read()
    if 'TFE_Data/' not in content:
        f.write("\n# Ignore Dataset Folder\nTFE_Data/\n")
        f.write("__pycache__/\n*.npy\n*.zip\n")
print(".gitignore security is active.")

Local Data Directory set to: /home/aysel/tfe/TFE_Data
Subdirectories created: Datasets/, Features_RAW/, Features_Reduced/, Results/
.gitignore security is active.


# Flikr8k

In [15]:
def build_flickr8k():
    print("\n--- BUILDING FLICKR8K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATASETS_DIR, 'df_Flickr8k.pkl')
    df.to_pickle(save_path) 
    
    print(f"Flickr8k Ready: {len(df)} images secured and saved to {save_path}.")
    return "Flickr8k"

# Flicker30k

In [16]:
def download_flickr30k_kaggle():
    print("\n--- DOWNLOADING FLICKR30K VIA KAGGLE ---")
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_8d2c016f8899a1eab14c99441d2bf6fe"
    flickr30k_dir = os.path.join(DATA_DIR, 'Flickr30k')
    os.makedirs(flickr30k_dir, exist_ok=True)

    print("Downloading Flickr30k (about 8 GB)...")
    dataset_cache_path = kagglehub.dataset_download("hsankesara/flickr-image-dataset")
    
    print(f"Download completed. Temp files found in: {dataset_cache_path}")
    print("Copying data to TFE_Data. Please wait...")
    shutil.copytree(dataset_cache_path, flickr30k_dir, dirs_exist_ok=True)
    print(f"Flickr30k raw files are ready in: {flickr30k_dir}")

def build_flickr30k():
    print("\n--- BUILDING FLICKR30K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr30k")
    img_dir = os.path.join(dataset_dir, "Images")
    csv_path = os.path.join(dataset_dir, "results.csv")
    
    if not os.path.exists(csv_path) or not os.path.exists(img_dir):
        print(f"Warning: Files not found in {dataset_dir}. Please run the Kaggle download first.")
        return None

    print("Reading the metadata file...")
    df_cap = pd.read_csv(csv_path, sep='|', on_bad_lines='skip') 
    if len(df_cap.columns) < 2:
        df_cap = pd.read_csv(csv_path, sep=',', on_bad_lines='skip')
        
    df_cap.columns = [str(col).strip() for col in df_cap.columns] 
    
    if 'image_name' not in df_cap.columns:
        print(f"Error: 'image_name' column does not exist. Columns found: {df_cap.columns.tolist()}")
        return None
    
    data = []
    for index, row in df_cap.iterrows():
        img_name = str(row['image_name']).strip()
        caption = str(row.get('comment', '')).strip() 
        full_path = os.path.join(img_dir, img_name)
        
        if os.path.exists(full_path) and caption != 'nan' and caption != '':
            data.append({
                "image_name": img_name, 
                "image_path": full_path, 
                "caption": caption
            })

    if len(data) == 0:
        print("\nFatal Error: 0 images found. Please check your image directory structure.")
        return None

    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATASETS_DIR, 'df_Flickr30k.pkl')
    df.to_pickle(save_path) 
    
    print(f"Flickr30k Ready: {len(df)} images secured and saved to {save_path}")
    return "Flickr30k"

# Conceptual Captions

In [17]:
def download_image_task(item):
    """Function executed by threads to download a single image."""
    index, url, caption, img_dir = item
    img_name = f"gcc_50k_{index}.jpg"
    full_path = os.path.join(img_dir, img_name)
    
    if os.path.exists(full_path):
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
        
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        response = requests.get(url, headers=headers, timeout=4)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        image.save(full_path, format="JPEG", quality=85)
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
    except:
        return {"success": False}

def build_conceptual_captions_50k(target_size=50000, max_workers=32):
    """Downloads exactly 50,000 valid images from the Conceptual Captions dataset."""
    print(f"\n--- BUILDING CONCEPTUAL CAPTIONS ({target_size} images) ---")
    dataset_dir = os.path.join(DATA_DIR, "ConceptualCaptions")
    img_dir = os.path.join(dataset_dir, "Images")
    os.makedirs(img_dir, exist_ok=True)
    
    try:
        from datasets import load_dataset
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "datasets"])
        from datasets import load_dataset

    print("1. Loading HuggingFace metadata...")
    ds = load_dataset("conceptual_captions", split="train")
    
    print("2. Selecting a pool of 100,000 URLs to account for dead links...")
    ds_subset = ds.select(range(100000))
    tasks = [(i, url, cap, img_dir) for i, (url, cap) in enumerate(zip(ds_subset['image_url'], ds_subset['caption']))]
        
    data = []
    successful_downloads = 0
    
    print(f"3. Starting download with {max_workers} threads...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_image_task, task): task for task in tasks}
        pbar = tqdm(total=target_size, desc="Valid images downloaded")
        
        for future in as_completed(futures):
            res = future.result()
            if res["success"]:
                data.append({
                    "image_name": res["image_name"],
                    "image_path": res["image_path"],
                    "caption": res["caption"]
                })
                successful_downloads += 1
                pbar.update(1)
                
            if successful_downloads >= target_size:
                print(f"\nTarget of {target_size} reached. Stopping current downloads...")
                for f in futures:
                    f.cancel()
                break
                
        pbar.close()

    print("\n4. Formatting and saving the DataFrame...")
    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATASETS_DIR, 'df_ConceptualCaptions.pkl')
    df.to_pickle(save_path) 
    
    print(f"Process complete. The dataset of {len(df)} images is ready in {save_path}")
    return "ConceptualCaptions"

# Execution of Downloading Flikr8k

In [20]:
# --- Execution ---
if __name__ == "__main__":
    # Uncomment the datasets you wish to build
    
    # 1. Flickr8k
    # build_flickr8k()
    
    # 2. Flickr30k
    # download_flickr30k_kaggle() 
    # build_flickr30k()
    
    # 3. Conceptual Captions
    # build_conceptual_captions_50k(target_size=50000, max_workers=32)
    pass


--- BUILDING FLICKR30K ---
Reading the metadata file...
Flickr30k Ready: 31783 images secured and saved to /home/aysel/tfe/TFE_Data/Datasets/df_Flickr30k.pkl
